In [3]:
from collections import defaultdict
import awkward as ak
import numba
import numpy as np
import pandas as pd
import h5py
import vector
vector.register_numba()
vector.register_awkward()

#import matplotlib.pyplot as plt
#from matplotlib.colors import LogNorm
#import mplhep as hep
#hep.style.use(hep.style.ROOT)

In [2]:
PARQUET_PATH = "/home/export/sdurgut/coffea_to_parquet_output/"

### First Step 
`python scripts/dataset/coffea_to_parquet.py -i input.coffea -o output_folder`

In [3]:
TTH_Hto2B_2022_postEE = ak.from_parquet(PARQUET_PATH + "output_TTH_Hto2B_2022_postEE.parquet")

In [ ]:
TTH_Hto2B_2022_postEE.fields

['JetGood', 'JetGoodMatched', 'Generator', 'MET', 'event']

: 

In [3]:
filename_test = "/home/export/sdurgut/scratch/ttHbb_SPANet/output_sig_0323_test_1876269.h5"
filename_pred = "/home/export/sdurgut/scratch/ttHbb_SPANet/predictions_full_signal_jet_assignment_version_2.h5"
df_test = h5py.File(filename_test,'r')
df_pred = h5py.File(filename_pred,'r')

In [17]:
df_pred["TARGETS"].keys()

<KeysViewHDF5 ['h', 't1', 't2']>

In [18]:
df_pred["TARGETS"]["h"].keys(), df_pred["TARGETS"]["t1"].keys(), df_pred["TARGETS"]["t2"].keys()

(<KeysViewHDF5 ['assignment_probability', 'b1', 'b2', 'detection_probability', 'marginal_probability']>,
 <KeysViewHDF5 ['assignment_probability', 'b', 'detection_probability', 'marginal_probability', 'q1', 'q2']>,
 <KeysViewHDF5 ['assignment_probability', 'b', 'detection_probability', 'marginal_probability', 'q1', 'q2']>)

In [5]:
idx_b1_pred = df_pred["TARGETS"]["h"]["b1"][()]
idx_b2_pred = df_pred["TARGETS"]["h"]["b2"][()]
idx_b1_pred

array([4, 6, 0, ..., 0, 1, 0], shape=(1876269,))

In [6]:
idx_b1_true = df_test["TARGETS"]["h"]["b1"][()]
idx_b2_true = df_test["TARGETS"]["h"]["b2"][()]
idx_b1_true

array([ 4,  2, -1, ..., -1,  3,  0], shape=(1876269,))

In [7]:
idx_h_pred = ak.concatenate((ak.unflatten(idx_b1_pred, ak.ones_like(idx_b1_pred)), ak.unflatten(idx_b2_pred, ak.ones_like(idx_b2_pred))), axis=1)
idx_h_true = ak.concatenate((ak.unflatten(idx_b1_true, ak.ones_like(idx_b1_true)), ak.unflatten(idx_b2_true, ak.ones_like(idx_b2_true))), axis=1)
idx_h_pred

<Array [[4, 6], [6, 6], ..., [1, 2], [0, 4]] type='1876269 * var * int64'>

In [8]:
idx_h_true

<Array [[4, 6], [2, -1], ..., [3, ...], [0, 4]] type='1876269 * var * int64'>

In [9]:
is_correct_higgs = ak.sum(idx_h_pred == idx_h_true, axis=1) == 2
is_correct_higgs

<Array [True, False, False, ..., False, False, True] type='1876269 * bool'>

In [10]:
n_tot = len(is_correct_higgs)
n_tot

1876269

In [11]:
n_correct = ak.sum(is_correct_higgs)
n_correct

np.int64(194228)

In [12]:
eff_h = n_correct / n_tot
eff_h

np.float64(0.1035182055451537)

In [13]:
idx_q1_pred = df_pred["TARGETS"]["t1"]["q1"][()]
idx_q2_pred = df_pred["TARGETS"]["t1"]["q2"][()]
idx_b_pred = df_pred["TARGETS"]["t1"]["b"][()]
idx_q1_true = df_test["TARGETS"]["t1"]["q1"][()]
idx_q2_true = df_test["TARGETS"]["t1"]["q2"][()]
idx_b_true = df_test["TARGETS"]["t1"]["b"][()]
idx_b_pred

array([1, 2, 5, ..., 4, 5, 1], shape=(1876269,))

In [14]:
idx_thad_pred = ak.concatenate(
     (ak.unflatten(idx_q1_pred, ak.ones_like(idx_q1_pred)),
     ak.unflatten(idx_q2_pred, ak.ones_like(idx_q2_pred)),
     ak.unflatten(idx_b_pred, ak.ones_like(idx_b_pred))),
     axis=1)
idx_thad_true = ak.concatenate(
     (ak.unflatten(idx_q1_true, ak.ones_like(idx_q1_true)),
     ak.unflatten(idx_q2_true, ak.ones_like(idx_q2_true)),
     ak.unflatten(idx_b_true, ak.ones_like(idx_b_true))),
     axis=1)
idx_thad_pred

<Array [[2, 7, 1], [1, 4, 2], ..., [3, 6, 1]] type='1876269 * var * int64'>

### Second Step \
`parquet_to_h5.py`

In [11]:
H5_FOLDER = "/home/export/sdurgut/scratch/parquet_to_h5_output/classification/with_MASK/"

In [ ]:
#open the h5 file
bkg_test = h5py.File(H5_FOLDER + 'output_sig_QCD_classification_0324_qcd_ht200_600_test_162481.h5', 'r')

In [21]:
sig_test = h5py.File(H5_FOLDER + 'output_sig_QCD_classification_0324_signal_only_test_1876269.h5', 'r')

In [22]:
bkg_test.keys(), sig_test.keys()

(<KeysViewHDF5 ['CLASSIFICATIONS', 'INPUTS', 'TARGETS', 'WEIGHTS']>,
 <KeysViewHDF5 ['CLASSIFICATIONS', 'INPUTS', 'TARGETS', 'WEIGHTS']>)

In [23]:
bkg_test['CLASSIFICATIONS'].keys(), sig_test['CLASSIFICATIONS'].keys()

(<KeysViewHDF5 ['EVENT']>, <KeysViewHDF5 ['EVENT']>)

In [24]:
bkg_test['CLASSIFICATIONS']['EVENT'].keys(), sig_test['CLASSIFICATIONS']['EVENT'].keys()

(<KeysViewHDF5 ['signal']>, <KeysViewHDF5 ['signal']>)

In [25]:
bkg_test['CLASSIFICATIONS']['EVENT']['signal'], sig_test['CLASSIFICATIONS']['EVENT']['signal']

(<HDF5 dataset "signal": shape (162481,), type "<i8">,
 <HDF5 dataset "signal": shape (1876269,), type "<i8">)

In [28]:
#print the values for bkg_test['CLASSIFICATIONS']['EVENT']['signal'], sig_test['CLASSIFICATIONS']['EVENT']['signal']
bkg_test['CLASSIFICATIONS']['EVENT']['signal'][:], sig_test['CLASSIFICATIONS']['EVENT']['signal'][:]

(array([0, 0, 0, ..., 0, 0, 0], shape=(162481,)),
 array([1, 1, 1, ..., 1, 1, 1], shape=(1876269,)))

In [30]:
bkg_test['INPUTS'].keys(), sig_test['INPUTS'].keys()

(<KeysViewHDF5 ['Event', 'Jet', 'Met']>,
 <KeysViewHDF5 ['Event', 'Jet', 'Met']>)

In [31]:
bkg_test['INPUTS']['Jet'].keys(), sig_test['INPUTS']['Jet'].keys()

(<KeysViewHDF5 ['MASK', 'btag', 'btag_CvB', 'btag_CvL', 'btag_L', 'btag_M', 'btag_QvG', 'btag_T', 'btag_XT', 'btag_XXT', 'eta', 'phi', 'pt']>,
 <KeysViewHDF5 ['MASK', 'btag', 'btag_CvB', 'btag_CvL', 'btag_L', 'btag_M', 'btag_QvG', 'btag_T', 'btag_XT', 'btag_XXT', 'eta', 'phi', 'pt']>)

In [32]:
bkg_test['INPUTS']['Met'].keys(), sig_test['INPUTS']['Met'].keys()

(<KeysViewHDF5 ['eta', 'm', 'phi', 'pt']>,
 <KeysViewHDF5 ['eta', 'm', 'phi', 'pt']>)

In [33]:
bkg_test['INPUTS']['Event'].keys(), sig_test['INPUTS']['Event'].keys()

(<KeysViewHDF5 ['ht']>, <KeysViewHDF5 ['ht']>)

In [34]:
#look at TARGETS 
bkg_test['TARGETS'].keys(), sig_test['TARGETS'].keys()

(<KeysViewHDF5 ['h', 't1', 't2']>, <KeysViewHDF5 ['h', 't1', 't2']>)

In [35]:
bkg_test['TARGETS']['h'].keys(), sig_test['TARGETS']['h'].keys()

(<KeysViewHDF5 ['b1', 'b2']>, <KeysViewHDF5 ['b1', 'b2']>)

In [36]:
bkg_test['TARGETS']['h']['b1'][:], sig_test['TARGETS']['h']['b1'][:]

(array([-1, -1, -1, ..., -1, -1, -1], shape=(162481,)),
 array([1, 2, 0, ..., 1, 5, 1], shape=(1876269,)))

In [39]:
sig_test['TARGETS']['h']['b1'][10:]

array([ 2,  0, -1, ...,  1,  5,  1], shape=(1876259,))